# 19_ensemble_deeponet_fno_bootstrap_val.ipynb

Стабильный выбор веса для ансамбля **DeepONet v2 + FNO v1** без утечки в `test`.

## Идея
Полностью корректный k-fold для ансамбля потребовал бы **переобучать базовые модели на каждом фолде**,
что слишком дорого. Поэтому здесь используется более дешёвый и честный компромисс:

- базовые модели фиксированы;
- предсказания считаются на `val`;
- вес ансамбля выбирается **только по `val`**;
- вместо одного single-split значения используется **bootstrap resampling** на `val`,
  чтобы сделать выбор веса более устойчивым;
- `test` используется только один раз — для финальной оценки уже выбранного веса.

Финальное предсказание:

\[
\hat y = w \cdot y_{DeepONet} + (1 - w) \cdot y_{FNO}
\]

## 0. Важно
`M_hom` считается через `j-Wave`, поэтому JAX запускается на **CPU**.

Обе модели используются только в режиме **inference**.

In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ.pop("LD_LIBRARY_PATH", None)

'/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64'

In [2]:
from pathlib import Path
import json
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu130
cuda available: True
device name: NVIDIA GeForce RTX 5090


In [3]:
CFG = {
    "dataset_root": "/workspace/usct_dataset_v2",
    "deeponet_ckpt": "/workspace/reference_points/deeponet_v2_main/deeponet_best.pt",
    "fno_ckpt": "/workspace/reference_points/fno_v1/fno_best.pt",
    "run_name": "ensemble_deeponet_fno_bootstrap_val",
    "mask_half_width": 2,
    "use_residual": True,

    "search_weights": [round(x, 2) for x in np.linspace(0.0, 1.0, 51)],
    "bootstrap_iters": 500,
    "bootstrap_seed": 123,
    "bootstrap_log_every": 25,
}
CFG


{'dataset_root': '/workspace/usct_dataset_v2',
 'deeponet_ckpt': '/workspace/reference_points/deeponet_v2_main/deeponet_best.pt',
 'fno_ckpt': '/workspace/reference_points/fno_v1/fno_best.pt',
 'run_name': 'ensemble_deeponet_fno_bootstrap_val',
 'mask_half_width': 2,
 'use_residual': True,
 'search_weights': [np.float64(0.0),
  np.float64(0.02),
  np.float64(0.04),
  np.float64(0.06),
  np.float64(0.08),
  np.float64(0.1),
  np.float64(0.12),
  np.float64(0.14),
  np.float64(0.16),
  np.float64(0.18),
  np.float64(0.2),
  np.float64(0.22),
  np.float64(0.24),
  np.float64(0.26),
  np.float64(0.28),
  np.float64(0.3),
  np.float64(0.32),
  np.float64(0.34),
  np.float64(0.36),
  np.float64(0.38),
  np.float64(0.4),
  np.float64(0.42),
  np.float64(0.44),
  np.float64(0.46),
  np.float64(0.48),
  np.float64(0.5),
  np.float64(0.52),
  np.float64(0.54),
  np.float64(0.56),
  np.float64(0.58),
  np.float64(0.6),
  np.float64(0.62),
  np.float64(0.64),
  np.float64(0.66),
  np.float64(0.68)

In [4]:
OUT_ROOT = Path(CFG["dataset_root"])
TRAIN_DIR = OUT_ROOT / "train"
VAL_DIR = OUT_ROOT / "val"
TEST_DIR = OUT_ROOT / "test"
DEEPO_CKPT_PATH = Path(CFG["deeponet_ckpt"])
FNO_CKPT_PATH = Path(CFG["fno_ckpt"])

assert TRAIN_DIR.exists(), TRAIN_DIR
assert VAL_DIR.exists(), VAL_DIR
assert TEST_DIR.exists(), TEST_DIR
assert DEEPO_CKPT_PATH.exists(), DEEPO_CKPT_PATH
assert FNO_CKPT_PATH.exists(), FNO_CKPT_PATH

train_files = sorted(TRAIN_DIR.glob("*.npz"))
val_files = sorted(VAL_DIR.glob("*.npz"))
test_files = sorted(TEST_DIR.glob("*.npz"))

print("train:", len(train_files))
print("val:", len(val_files))
print("test:", len(test_files))
print("DeepONet ckpt:", DEEPO_CKPT_PATH)
print("FNO ckpt:", FNO_CKPT_PATH)

train: 6200
val: 1000
test: 800
DeepONet ckpt: /workspace/reference_points/deeponet_v2_main/deeponet_best.pt
FNO ckpt: /workspace/reference_points/fno_v1/fno_best.pt


In [5]:
sample0 = np.load(train_files[0])

S, R = sample0["x"].shape[1], sample0["x"].shape[2]
H, W = sample0["c"].shape
dx = tuple(sample0["dx"].astype(np.float32))
freq_hz = float(sample0["freq_hz"][0])

receiver_y = np.array(sample0["receiver_y"].astype(np.int32))
receiver_x = np.array(sample0["receiver_x"].astype(np.int32))
source_ring_idx_full = np.array(sample0["source_ring_idx"].astype(np.int32))

all_c_values = []
for p in train_files:
    c = np.load(p)["c"].astype(np.float32)
    all_c_values.append(c)
all_c_values = np.stack(all_c_values, axis=0)

stats = {
    "c_min": float(all_c_values.min()),
    "c_max": float(all_c_values.max()),
}
print("S, R:", S, R)
print("H, W:", H, W)
print("dx:", dx)
print("freq_hz:", freq_hz)
print("stats:", stats)

S, R: 16 256
H, W: 192 192
dx: (np.float32(0.00075), np.float32(0.00075))
freq_hz: 500000.0
stats: {'c_min': 1404.1473388671875, 'c_max': 1598.34423828125}


## 1. `M_hom` через j-Wave

In [6]:
import jax
import jax.numpy as jnp
from jax import jit

from jwave import FourierSeries
from jwave.geometry import Domain, Medium
from jwave.acoustics.time_harmonic import helmholtz_solver

print("jax version:", jax.__version__)
print("jax devices:", jax.devices())

jax version: 0.4.38
jax devices: [CudaDevice(id=0)]


In [7]:
domain = Domain((H, W), dx)
omega = 2 * np.pi * freq_hz

def gaussian_source(H, W, y0, x0, sigma=1.8):
    yy, xx = np.mgrid[0:H, 0:W]
    g = np.exp(-((yy - y0) ** 2 + (xx - x0) ** 2) / (2 * sigma**2))
    g = g / g.sum()
    return g.astype(np.float32)

full_source_fields = []
for ridx in source_ring_idx_full:
    y0 = int(receiver_y[ridx])
    x0 = int(receiver_x[ridx])
    src_np = gaussian_source(H, W, y0, x0, sigma=1.8)
    full_source_fields.append(jnp.array(src_np, dtype=jnp.complex64))
full_source_fields = tuple(full_source_fields)

def make_medium(c_map):
    sound_speed = FourierSeries(jnp.expand_dims(jnp.array(c_map), -1), domain)
    medium = Medium(domain=domain, sound_speed=sound_speed, density=1000.0, pml_size=16)
    return medium

def forward_measurements_full(c_map):
    medium = make_medium(c_map)
    ms = []
    for i in range(len(full_source_fields)):
        src = FourierSeries(jnp.expand_dims(full_source_fields[i], -1), domain)
        field = helmholtz_solver(medium, omega, src * omega)
        u = field.params[..., 0]
        ms.append(u[receiver_y, receiver_x])
    return jnp.stack(ms, axis=0)

forward_measurements_full_jit = jit(forward_measurements_full)

/tmp/ipykernel_317173/1657452440.py:1: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  domain = Domain((H, W), dx)


In [8]:
c_hom = jnp.full((H, W), 1500.0, dtype=jnp.float32)

t0 = time.time()
M_hom_complex = forward_measurements_full_jit(c_hom)
M_hom_complex.block_until_ready()
t1 = time.time()

M_hom_complex = np.array(M_hom_complex)

print("M_hom_complex:", M_hom_complex.shape, M_hom_complex.dtype)
print("time:", t1 - t0, "sec")

M_hom_complex: (16, 256) complex64
time: 7.093070983886719 sec


## 2. Dataset

In [9]:
def mask_self_receivers(M, source_ring_idx, half_width=2):
    M = M.copy()
    S_local, R_local = M.shape
    for s in range(S_local):
        ridx = int(source_ring_idx[s])
        for delta in range(-half_width, half_width + 1):
            rr = (ridx + delta) % R_local
            M[s, rr] = 0.0 + 0.0j
    return M

class USCTEnsembleDataset(Dataset):
    def __init__(self, paths, c_min, c_max, M_hom_complex, use_residual=True, mask_half_width=2):
        self.paths = list(paths)
        self.c_min = float(c_min)
        self.c_max = float(c_max)
        self.c_range = max(self.c_max - self.c_min, 1e-8)
        self.M_hom_complex = M_hom_complex
        self.use_residual = bool(use_residual)
        self.mask_half_width = int(mask_half_width)

    def __len__(self):
        return len(self.paths)

    def normalize_target(self, c):
        return 2.0 * (c - self.c_min) / self.c_range - 1.0

    def denormalize_target(self, c_norm):
        return ((c_norm + 1.0) * 0.5) * self.c_range + self.c_min

    def __getitem__(self, idx):
        d = np.load(self.paths[idx])

        x_raw = d["x"].astype(np.float32)
        c = d["c"].astype(np.float32)
        source_ring_idx = d["source_ring_idx"].astype(np.int64)

        M = x_raw[0] + 1j * x_raw[1]

        if self.use_residual:
            M = M - self.M_hom_complex

        if self.mask_half_width > 0:
            M = mask_self_receivers(M, source_ring_idx, half_width=self.mask_half_width)

        x_tensor = np.stack([M.real, M.imag], axis=0).astype(np.float32)

        x_scale = float(np.percentile(np.abs(x_tensor), 99.5))
        if x_scale < 1e-8:
            x_scale = 1.0
        x_tensor = x_tensor / x_scale

        c_norm = self.normalize_target(c).astype(np.float32)

        return (
            torch.from_numpy(x_tensor),
            torch.from_numpy(c_norm).unsqueeze(0),
            {"path": str(self.paths[idx]), "x_scale": x_scale},
        )

In [10]:
ds_val = USCTEnsembleDataset(val_files, stats["c_min"], stats["c_max"], M_hom_complex=M_hom_complex, use_residual=CFG["use_residual"], mask_half_width=CFG["mask_half_width"])
ds_test = USCTEnsembleDataset(test_files, stats["c_min"], stats["c_max"], M_hom_complex=M_hom_complex, use_residual=CFG["use_residual"], mask_half_width=CFG["mask_half_width"])

x0, c0, meta0 = ds_val[0]
print("x0:", x0.shape, x0.dtype)
print("c0:", c0.shape, c0.dtype, float(c0.min()), float(c0.max()))
print(meta0)

x0: torch.Size([2, 16, 256]) torch.float32
c0: torch.Size([1, 192, 192]) torch.float32 -0.6515107154846191 0.5998921394348145
{'path': '/workspace/usct_dataset_v2/val/sample_00000.npz', 'x_scale': 5099.13671875}


## 3. Model definitions

In [11]:
class BranchEncoderV2(nn.Module):
    def __init__(self, out_dim=192, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(1, 32),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.GroupNorm(1, 64),
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.GroupNorm(1, 128),
            nn.GELU(),
            nn.Conv2d(128, 192, kernel_size=3, stride=2, padding=1),
            nn.GroupNorm(1, 192),
            nn.GELU(),
            nn.Flatten(),
            nn.Linear(192 * 2 * 32, 384),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(384, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class TrunkNetV2(nn.Module):
    def __init__(self, width=192, hidden=256, num_fourier_features=24, dropout=0.0):
        super().__init__()
        self.num_fourier_features = int(num_fourier_features)
        if self.num_fourier_features > 0:
            freq = 2.0 ** torch.arange(self.num_fourier_features, dtype=torch.float32) * math.pi
            self.register_buffer("freq", freq, persistent=False)
            in_dim = 2 + 4 * self.num_fourier_features
        else:
            self.register_buffer("freq", torch.empty(0), persistent=False)
            in_dim = 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, width),
        )

    def encode_coords(self, coords):
        if self.num_fourier_features == 0:
            return coords
        y = coords[:, 0:1]
        x = coords[:, 1:2]
        y_proj = y * self.freq[None, :]
        x_proj = x * self.freq[None, :]
        feats = [coords, torch.sin(y_proj), torch.cos(y_proj), torch.sin(x_proj), torch.cos(x_proj)]
        return torch.cat(feats, dim=1)

    def forward(self, coords):
        return self.net(self.encode_coords(coords))

class DeepONet2DV2(nn.Module):
    def __init__(self, branch_width=192, trunk_width=192, trunk_hidden=256, num_fourier_features=24, dropout=0.0):
        super().__init__()
        self.width = int(branch_width)
        self.branch = BranchEncoderV2(out_dim=branch_width, dropout=dropout)
        self.trunk = TrunkNetV2(width=trunk_width, hidden=trunk_hidden, num_fourier_features=num_fourier_features, dropout=dropout)
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x, coords):
        b = self.branch(x)
        t = self.trunk(coords)
        out = torch.matmul(b, t.t()) / math.sqrt(self.width) + self.bias
        return out.view(x.shape[0], 1, H, W)

In [12]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        scale = 1 / (in_channels * out_channels)
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.weights1 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        x = x.float()
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft2(x)
        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-2), x.size(-1)//2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2] = self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        return torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))

class FNOBlock2d(nn.Module):
    def __init__(self, width, modes1, modes2):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes1, modes2)
        self.pointwise = nn.Conv2d(width, width, kernel_size=1)
    def forward(self, x):
        return F.gelu(self.spectral(x) + self.pointwise(x))

class FNO2dBaseline(nn.Module):
    def __init__(self, width=48, depth=4, modes1=24, modes2=24, padding=8):
        super().__init__()
        self.padding = padding
        self.input_proj = nn.Conv2d(4, width, kernel_size=1)
        self.blocks = nn.ModuleList([FNOBlock2d(width, modes1, modes2) for _ in range(depth)])
        self.head = nn.Sequential(nn.Conv2d(width, width, kernel_size=1), nn.GELU(), nn.Conv2d(width, 1, kernel_size=1))

    def get_grid(self, batch_size, device):
        yy = torch.linspace(-1.0, 1.0, H, device=device)
        xx = torch.linspace(-1.0, 1.0, W, device=device)
        gy, gx = torch.meshgrid(yy, xx, indexing="ij")
        return torch.stack([gy, gx], dim=0).unsqueeze(0).repeat(batch_size, 1, 1, 1)

    def forward(self, x):
        x = F.interpolate(x, size=(H, W), mode="bilinear", align_corners=False)
        grid = self.get_grid(x.shape[0], x.device)
        x = torch.cat([x, grid], dim=1)
        x = self.input_proj(x)
        if self.padding > 0:
            x = F.pad(x, [0, self.padding, 0, self.padding])
        for block in self.blocks:
            x = block(x)
        if self.padding > 0:
            x = x[..., :H, :W]
        return self.head(x)

## 4. Load checkpoints and cache predictions

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deeponet_ckpt = torch.load(DEEPO_CKPT_PATH, map_location=device)
fno_ckpt = torch.load(FNO_CKPT_PATH, map_location=device)

deeponet_cfg = deeponet_ckpt["config"]
fno_cfg = fno_ckpt["config"]

yy = np.linspace(-1.0, 1.0, H, dtype=np.float32)
xx = np.linspace(-1.0, 1.0, W, dtype=np.float32)
grid_y, grid_x = np.meshgrid(yy, xx, indexing="ij")
coords = np.stack([grid_y, grid_x], axis=-1).reshape(-1, 2)
coords_t = torch.from_numpy(coords).to(device)

deeponet = DeepONet2DV2(
    branch_width=deeponet_cfg["branch_width"],
    trunk_width=deeponet_cfg["trunk_width"],
    trunk_hidden=deeponet_cfg["trunk_hidden"],
    num_fourier_features=deeponet_cfg["num_fourier_features"],
    dropout=deeponet_cfg["dropout"],
).to(device)
deeponet.load_state_dict(deeponet_ckpt["model_state_dict"], strict=True)
deeponet.eval()

fno = FNO2dBaseline(
    width=fno_cfg["fno_width"],
    depth=fno_cfg["fno_depth"],
    modes1=fno_cfg["fno_modes_h"],
    modes2=fno_cfg["fno_modes_w"],
    padding=fno_cfg["fno_padding"],
).to(device)
fno.load_state_dict(fno_ckpt["model_state_dict"], strict=True)
fno.eval()

print("DeepONet best epoch:", deeponet_ckpt.get("best_epoch"))
print("FNO best epoch:", fno_ckpt.get("best_epoch"))

DeepONet best epoch: 197
FNO best epoch: 159


In [14]:
def denorm_c_from_minus1_1(c_norm, c_min, c_max):
    return ((c_norm + 1.0) * 0.5) * (c_max - c_min) + c_min

@torch.no_grad()
def collect_predictions(dataset):
    preds_deepo = []
    preds_fno = []
    targets = []
    paths = []

    for idx in range(len(dataset)):
        x, c, meta = dataset[idx]
        xb = x.unsqueeze(0).to(device)
        pd = deeponet(xb, coords_t)[0, 0].cpu().numpy()
        pf = fno(xb)[0, 0].cpu().numpy()

        pd = np.clip(pd, -1.0, 1.0)
        pf = np.clip(pf, -1.0, 1.0)
        tg = c[0].numpy()

        preds_deepo.append(pd.astype(np.float32))
        preds_fno.append(pf.astype(np.float32))
        targets.append(tg.astype(np.float32))
        paths.append(meta["path"])

    preds_deepo = np.stack(preds_deepo, axis=0)
    preds_fno = np.stack(preds_fno, axis=0)
    targets = np.stack(targets, axis=0)
    return preds_deepo, preds_fno, targets, paths

val_pred_deepo, val_pred_fno, val_targets, val_paths = collect_predictions(ds_val)
test_pred_deepo, test_pred_fno, test_targets, test_paths = collect_predictions(ds_test)

print("val_pred_deepo:", val_pred_deepo.shape)
print("test_pred_deepo:", test_pred_deepo.shape)

val_pred_deepo: (1000, 192, 192)
test_pred_deepo: (800, 192, 192)


## 5. Bootstrap weight search on validation

In [ ]:
def rmse_array(pred_phys, target_phys):
    return np.sqrt(np.mean((pred_phys - target_phys) ** 2, axis=(1, 2)))

rng = np.random.default_rng(CFG["bootstrap_seed"])

weights = np.array(CFG["search_weights"], dtype=np.float32)
val_rmse_per_weight = []
best_weights_each_bootstrap = []

n_val = len(val_targets)
t_boot_0 = time.time()

base_all = val_pred_fno - val_targets
diff_all = val_pred_deepo - val_pred_fno

A_all = np.mean(diff_all * diff_all, axis=(1, 2))
B_all = 2.0 * np.mean(base_all * diff_all, axis=(1, 2))
C_all = np.mean(base_all * base_all, axis=(1, 2))

for b in range(CFG["bootstrap_iters"]):
    print("current stage ", b)
    idx = rng.integers(0, n_val, size=n_val)

    tgt = val_targets[idx]
    pd = val_pred_deepo[idx]
    pf = val_pred_fno[idx]

    A = A_all[idx]
    B = B_all[idx]
    C = C_all[idx]
    
    rmse_grid = []
    for w in weights:
        mse_per_sample = A * (w ** 2) + B * w + C
        mse_per_sample = np.maximum(mse_per_sample, 0.0)
        rmse_grid.append(float(np.sqrt(mse_per_sample).mean()))
    rmse_grid = np.array(rmse_grid, dtype=np.float32)
    val_rmse_per_weight.append(rmse_grid)

    best_w_this = float(weights[np.argmin(rmse_grid)])
    best_weights_each_bootstrap.append(best_w_this)

    log_every = int(CFG.get("bootstrap_log_every", 25))
    if (b == 0) or ((b + 1) % log_every == 0) or (b + 1 == CFG["bootstrap_iters"]):
        elapsed = time.time() - t_boot_0
        avg_per_iter = elapsed / (b + 1)
        eta = avg_per_iter * (CFG["bootstrap_iters"] - (b + 1))

        cur_stack = np.stack(val_rmse_per_weight, axis=0)
        cur_mean_rmse = cur_stack.mean(axis=0)
        cur_best_idx = int(np.argmin(cur_mean_rmse))
        cur_best_w = float(weights[cur_best_idx])

        print(
            f"bootstrap {b+1}/{CFG['bootstrap_iters']} | "
            f"last_best_w={best_w_this:.2f} | "
            f"current_mean_best_w={cur_best_w:.2f} | "
            f"elapsed={elapsed:.1f}s | eta={eta:.1f}s"
        )

val_rmse_per_weight = np.stack(val_rmse_per_weight, axis=0)
best_weights_each_bootstrap = np.array(best_weights_each_bootstrap, dtype=np.float32)

mean_rmse = val_rmse_per_weight.mean(axis=0)
std_rmse = val_rmse_per_weight.std(axis=0)

best_idx = int(np.argmin(mean_rmse))
best_w_bootstrap = float(weights[best_idx])

single_split_rmse = []
for w in weights:
    pred = w * val_pred_deepo + (1.0 - w) * val_pred_fno
    pred = np.clip(pred, -1.0, 1.0)
    pred_phys = denorm_c_from_minus1_1(pred, stats["c_min"], stats["c_max"])
    tgt_phys = denorm_c_from_minus1_1(val_targets, stats["c_min"], stats["c_max"])
    single_split_rmse.append(float(np.mean(rmse_array(pred_phys, tgt_phys))))
single_split_rmse = np.array(single_split_rmse, dtype=np.float32)
best_w_single = float(weights[int(np.argmin(single_split_rmse))])

print("best bootstrap weight:", best_w_bootstrap)
print("best single-split val weight:", best_w_single)
print("bootstrap best-weight mean:", float(best_weights_each_bootstrap.mean()))
print("bootstrap best-weight median:", float(np.median(best_weights_each_bootstrap)))
print("bootstrap best-weight std:", float(best_weights_each_bootstrap.std()))


current stage  0


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(weights, mean_rmse, marker="o", label="bootstrap mean val RMSE")
plt.fill_between(weights, mean_rmse - std_rmse, mean_rmse + std_rmse, alpha=0.2, label="±1 std")
plt.axvline(best_w_bootstrap, linestyle="--", label=f"bootstrap best={best_w_bootstrap:.2f}")
plt.axvline(best_w_single, linestyle=":", label=f"single val best={best_w_single:.2f}")
plt.xlabel("weight on DeepONet")
plt.ylabel("validation RMSE")
plt.title("Bootstrap-stable ensemble weight search")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(best_weights_each_bootstrap, bins=len(weights), edgecolor="black")
plt.xlabel("best weight on DeepONet across bootstraps")
plt.ylabel("count")
plt.title("Distribution of bootstrap-optimal weights")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Final evaluation on test with bootstrap-selected weight

In [ ]:
def compute_metrics(pred, target, data_range):
    mse = float(np.mean((pred - target) ** 2))
    rmse = float(np.sqrt(mse))
    rel = float(np.linalg.norm(pred - target) / (np.linalg.norm(target) + 1e-8))
    psnr = float(20 * np.log10(data_range) - 10 * np.log10(mse)) if mse > 1e-12 else float("inf")
    out = {
        "mse": mse,
        "rmse": rmse,
        "relative_l2": rel,
        "psnr": psnr,
    }
    try:
        from skimage.metrics import structural_similarity as ssim_fn
        out["ssim"] = float(ssim_fn(pred, target, data_range=data_range))
    except Exception:
        pass
    return out

def evaluate_cached(pred_deepo, pred_fno, targets, paths, weight):
    data_range = float(stats["c_max"] - stats["c_min"])
    metrics_list = []
    for i in range(len(targets)):
        pred = weight * pred_deepo[i] + (1.0 - weight) * pred_fno[i]
        pred = np.clip(pred, -1.0, 1.0)

        pred_phys = denorm_c_from_minus1_1(pred, stats["c_min"], stats["c_max"])
        tgt_phys = denorm_c_from_minus1_1(targets[i], stats["c_min"], stats["c_max"])

        m = compute_metrics(pred_phys, tgt_phys, data_range=data_range)
        m["index"] = i
        m["path"] = paths[i]
        metrics_list.append(m)
    return metrics_list

def summarize_metrics(metrics_list):
    summary = {}
    for key in metrics_list[0].keys():
        if key in ("index", "path"):
            continue
        vals = [m[key] for m in metrics_list]
        summary[f"{key}_mean"] = float(np.mean(vals))
        summary[f"{key}_std"] = float(np.std(vals))
    return summary

val_metrics_bootstrap = evaluate_cached(val_pred_deepo, val_pred_fno, val_targets, val_paths, best_w_bootstrap)
test_metrics_bootstrap = evaluate_cached(test_pred_deepo, test_pred_fno, test_targets, test_paths, best_w_bootstrap)

val_summary_bootstrap = summarize_metrics(val_metrics_bootstrap)
test_summary_bootstrap = summarize_metrics(test_metrics_bootstrap)

print("SELECTED BOOTSTRAP WEIGHT")
print(json.dumps({"weight_deeponet": best_w_bootstrap, "weight_fno": 1.0 - best_w_bootstrap}, ensure_ascii=False, indent=2))

print("\nVAL SUMMARY (bootstrap-selected)")
print(json.dumps(val_summary_bootstrap, ensure_ascii=False, indent=2))

print("\nTEST SUMMARY (bootstrap-selected)")
print(json.dumps(test_summary_bootstrap, ensure_ascii=False, indent=2))

## 7. Visualization

In [ ]:
def show_prediction(idx, weight):
    pred_deepo = val_pred_deepo[idx] if idx < len(val_targets) else None

@torch.no_grad()
def show_test_prediction(idx, weight):
    pred_deepo = test_pred_deepo[idx]
    pred_fno = test_pred_fno[idx]
    target = test_targets[idx]

    pred = weight * pred_deepo + (1.0 - weight) * pred_fno
    pred = np.clip(pred, -1.0, 1.0)

    pred_deepo_phys = denorm_c_from_minus1_1(pred_deepo, stats["c_min"], stats["c_max"])
    pred_fno_phys = denorm_c_from_minus1_1(pred_fno, stats["c_min"], stats["c_max"])
    pred_phys = denorm_c_from_minus1_1(pred, stats["c_min"], stats["c_max"])
    target_phys = denorm_c_from_minus1_1(target, stats["c_min"], stats["c_max"])

    fig, ax = plt.subplots(1, 5, figsize=(25, 4))

    im0 = ax[0].imshow(target_phys, origin="lower")
    ax[0].set_title(f"target | idx={idx}")
    plt.colorbar(im0, ax=ax[0])

    im1 = ax[1].imshow(pred_deepo_phys, origin="lower")
    ax[1].set_title("DeepONet v2")
    plt.colorbar(im1, ax=ax[1])

    im2 = ax[2].imshow(pred_fno_phys, origin="lower")
    ax[2].set_title("FNO v1")
    plt.colorbar(im2, ax=ax[2])

    im3 = ax[3].imshow(pred_phys, origin="lower")
    ax[3].set_title(f"bootstrap ensemble (w={weight:.2f})")
    plt.colorbar(im3, ax=ax[3])

    im4 = ax[4].imshow(np.abs(pred_phys - target_phys), origin="lower")
    ax[4].set_title("|ensemble-target|")
    plt.colorbar(im4, ax=ax[4])

    plt.tight_layout()
    plt.show()
    print(test_paths[idx])

for idx in [0, 1, 2]:
    show_test_prediction(idx, best_w_bootstrap)

## 8. Save artifacts

In [ ]:
RUN_DIR = Path("/workspace/reference_points") / CFG["run_name"]
RUN_DIR.mkdir(parents=True, exist_ok=True)

with open(RUN_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "config": CFG,
            "stats": stats,
            "S": int(S),
            "R": int(R),
            "H": int(H),
            "W": int(W),
            "dx": [float(v) for v in dx],
            "freq_hz": float(freq_hz),
            "selected_weight_deeponet_bootstrap": float(best_w_bootstrap),
            "selected_weight_fno_bootstrap": float(1.0 - best_w_bootstrap),
            "selected_weight_deeponet_single_val": float(best_w_single),
            "selected_weight_fno_single_val": float(1.0 - best_w_single),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(RUN_DIR / "bootstrap_weight_distribution.json", "w", encoding="utf-8") as f:
    json.dump(best_weights_each_bootstrap.tolist(), f, ensure_ascii=False, indent=2)

with open(RUN_DIR / "bootstrap_rmse_curve.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "weights": weights.tolist(),
            "mean_rmse": mean_rmse.tolist(),
            "std_rmse": std_rmse.tolist(),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(RUN_DIR / "val_summary_bootstrap.json", "w", encoding="utf-8") as f:
    json.dump(val_summary_bootstrap, f, ensure_ascii=False, indent=2)
with open(RUN_DIR / "test_summary_bootstrap.json", "w", encoding="utf-8") as f:
    json.dump(test_summary_bootstrap, f, ensure_ascii=False, indent=2)

with open(RUN_DIR / "val_metrics_bootstrap.json", "w", encoding="utf-8") as f:
    json.dump(val_metrics_bootstrap, f, ensure_ascii=False, indent=2)
with open(RUN_DIR / "test_metrics_bootstrap.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics_bootstrap, f, ensure_ascii=False, indent=2)

print("saved to:", RUN_DIR)